# Civic Comments: Estimate-Level Adjustment Results

This notebook applies the estimate-level adjustment methodology to the Civil Comments 
public dataset. It demonstrates coverage calibration and domain-level metric correction.

## Prerequisites
- Run `train_toxicity_model.ipynb` first to generate `civil_comments_metrics.csv` and `civil_comments_entities.csv`

In [ ]:
import numpy as np
import pandas as pd
from estimate_level_adjustment.adjustment_model import em_latent_normal
from estimate_level_adjustment.utils import (
    coverage_calibration_curve,
    leave_one_out_validation,
)
from estimate_level_adjustment.summarize_results import (
    adapt_stats_to_coverage,
    clean_domain_stats,
    compute_coefficient_mean,
    plot_coverage_curve,
    plot_coverage_and_domain_metrics_side_by_side,
    plot_domain_level_metrics,
)

## Load Data

Load the metrics and entities CSVs generated by the training notebook, then join them into a single DataFrame with primary and proxy metrics per row.

In [ ]:
# Load saved CSV files from training notebook
metrics_df = pd.read_csv("civil_comments_metrics.csv")
entities_df = pd.read_csv("civil_comments_entities.csv")

# Pivot metrics to wide format and join with entities
proxy_df = metrics_df[metrics_df["model_name"] == "proxy"][["row_id", "metric"]].rename(
    columns={"metric": "proxy_metric"}
)
primary_df = metrics_df[metrics_df["model_name"] == "primary"][["row_id", "metric"]].rename(
    columns={"metric": "primary_metric"}
)
df_tiger_tables = proxy_df.merge(primary_df, on="row_id").merge(entities_df, on="row_id")
df_tiger_tables.head()

## Overall Statistics

In [ ]:
domain_column = None
stats = compute_coefficient_mean(df_tiger_tables, domain_column)
stats = clean_domain_stats(stats, minimum_count=10)
stats

## Domain-Level Analysis (by publication_id)

In [ ]:
domain_column = ["publication_id"]
alpha_seq = np.linspace(1, 0.005, 200)
stats = compute_coefficient_mean(df_tiger_tables, domain_column)
stats = clean_domain_stats(stats, minimum_count=10)
stats = stats[stats["primary_mean"] > 0]
stats['count'].sort_values(ascending=False)

In [ ]:
(
    mean_primary,
    sigma_primary,
    mean_proxy,
    sigma_proxy,
    mean_proxy_corr,
    sigma_proxy_corr,
) = adapt_stats_to_coverage(stats)
base_coverage, corr_coverage, alpha_seq = coverage_calibration_curve(
    mean_primary,
    sigma_primary,
    mean_proxy,
    sigma_proxy,
    mean_proxy_corr,
    sigma_proxy_corr,
    alpha_seq=alpha_seq,
)

## Coverage at alpha = 0.05

In [ ]:
overlap_res = pd.DataFrame([base_coverage, corr_coverage, alpha_seq]).transpose()
overlap_res.columns = ['base_coverage', 'corr_coverage', 'alpha_seq']
overlap_res[(overlap_res['alpha_seq'] <= 0.050) & (overlap_res['alpha_seq'] >= 0.040)]

## CI Length Comparison

In [ ]:
ci_length_res = pd.DataFrame([sigma_primary, sigma_proxy, sigma_proxy_corr]).transpose()
ci_length_res.columns = ['sigma_primary', 'sigma_proxy', 'sigma_proxy_corr']
primary_average_ci_length = ci_length_res['sigma_primary'].mean()
proxy_average_ci_length = ci_length_res['sigma_proxy'].mean()
proxy_corr_average_ci_length = ci_length_res['sigma_proxy_corr'].mean()
print(f"Proxy/Primary CI ratio: {proxy_average_ci_length/primary_average_ci_length:.4f}")
print(f"Corrected Proxy/Primary CI ratio: {proxy_corr_average_ci_length/primary_average_ci_length:.4f}")

## Plots

In [ ]:
plot_coverage_curve(base_coverage, corr_coverage, alpha_seq)

In [ ]:
plot_domain_level_metrics(stats)

In [ ]:
plot_coverage_and_domain_metrics_side_by_side(
    base_coverage, corr_coverage, alpha_seq, stats
)